AWS AI Practitioner Exam Preparation Assistant
This notebook builds an AI-powered assistant to help users prepare for the AWS AI Practitioner exam. It leverages LlamaIndex, Wolfram Alpha, CrewAI, and LangChain to provide comprehensive and accurate responses to user queries.



# Installation of Required Packages

The following commands install the necessary packages for our AWS AI Practitioner Exam Preparation Assistant. These packages provide support for:

- **llama-index**: A set of libraries for indexing and querying large language models.
- **Wolfram Alpha Tools**: Tools for integrating Wolfram Alpha for advanced computation.
- **CrewAI Tools**: Tools used by the CrewAI framework for agent-based systems.
- **LangChain Integr**


In [ ]:
%pip install llama-index-core
%pip install llama-index-readers-file
%pip install llama-index-tools-wolfram-alpha
%pip install crewai[tools]
%pip install llama-index-embeddings-langchain
%pip install llama-index-llms-groq
%pip install langchain_groq
%pip install llama-index-llms-langchain

# Importing Required Libraries and Tools

This cell imports the necessary modules and tools for building our AWS AI Practitioner Exam Preparation Assistant. Here's a breakdown of the key imports:

- **llama_index Modules:**
  - `from llama_index.llms.groq import Groq`: Imports the Groq LLM module which can be used for specialized language model operations.
  - `from llama_index.llms.openai import OpenAI`: Provides integration with OpenAI's language models.
  - `from llama_index.core import SimpleDirectoryReader, VectorStoreIndex`:
    - `SimpleDirectoryReader` helps to read and parse documents from directories.
    - `VectorStoreIndex` is used to create and query a vector index for efficient document retrieval.
  - `from llama_index.embeddings.huggingface import HuggingFaceEmbedding`: Imports embeddings from HuggingFace, useful for generating vector representations of text.

- **langchain_openai:**
  - `from langchain_openai import ChatOpenAI_


In [43]:
from llama_index.llms.groq import Groq
from langchain_openai import ChatOpenAI
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex
from llama_index.llms.openai import OpenAI
from langchain_openai import ChatOpenAI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from crewai_tools import LlamaIndexTool
from crewai_tools import ScrapeWebsiteTool
from crewai import LLM
from crewai import Agent, Task, Crew, Process

# LLM Initialization

The LLM (Language Model) is a core component that powers our assistant's ability to generate dynamic, context-aware responses. In our setup, we initialize the LLM using specific parameters such as the model name, temperature, API base URL, and API key.

### Key Parameters:
- **model**: Specifies which language model to use (e.g., "gpt-3.5-turbo"). This determines the capabilities and performance of the generated responses.
- **temperature**: Controls the randomness of the output. A higher temperature (e.g., 0.7) leads to more diverse and creative responses, while a lower value produces more deterministic outputs.
- **base_url**: The endpoint URL for the API. This is where requests are sent.
- **api_key**: A unique key used to authenticate your requests to the API.



In [44]:

llm_agent = LLM(
    model="gpt-3.5-turbo",
    temperature=0.7,
    base_url="https://api.nexus.navigatelabsai.com",
    api_key="sk-3yaE7AQUcjAqByOQ_TIx2A"
)

Embeddings Initialization

In [47]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small", openai_api_base="https://api.nexus.navigatelabsai.com", openai_api_key="sk-QjjMoRwKPjHXTEzvcum3eA")

text = "This is a test document."

query_result = embeddings.embed_query(text)


Data Loading and Indexing

In [48]:

# To read data from input files
reader = SimpleDirectoryReader(input_files=["/content/AWS-Certified-AI-Practitioner_Exam-Guide (1).pdf"])
docs = reader.load_data()


Query Engine and Tools

In [49]:
#VectorStoreIndex with documents and embedding model
index = VectorStoreIndex.from_documents(docs,embed_model=embeddings,)

/usr/local/lib/python3.11/dist-packages/llama_index/core/indices/base.py:110: DeprecationWarning: Call to deprecated method get_doc_id. ('get_doc_id' is deprecated, access the 'id_' property instead.) -- Deprecated since version 0.12.2.
  docstore.set_document_hash(doc.get_doc_id(), doc.hash)


In [50]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(temperature=0.7, model_name="gpt-3.5-turbo", openai_api_base="https://api.nexus.navigatelabsai.com", openai_api_key="sk-QjjMoRwKPjHXTEzvcum3eA")

In [51]:

#This sets up the query engine to retrieve the top 5 most similar documents using the specified large language model (LLM).
query_engine = index.as_query_engine(similarity_top_k=5, llm=llm)

In [52]:
query_tool = LlamaIndexTool.from_query_engine(
    query_engine,
    name="AWS AI practioner exam guide",
    description="Use this tool to lookup the aws ai practioner exam quide",
)


In [53]:
# To enable scrapping any website it finds during it's execution
scrape_tool = ScrapeWebsiteTool()


Agent, Task, and Crew Definition

In [54]:
aws_ai_exam_agent = Agent(
    role="AWS AI Practitioner Exam Assistant",
    goal="Assist users in preparing for the AWS AI Practitioner exam by providing accurate responses based on the official exam syllabus, while supplementing with general knowledge about exam-relevant topics.",
    backstory=(
        "An AI-powered AWS AI Practitioner exam assistant designed to help candidates prepare effectively. "
        "It first consults the official exam syllabus and structure using the query tool. "
        "Additionally, it utilizes the scrape tool to retrieve general knowledge about exam topics—strictly from reputable certification-related sources. "
        "However, the scrape tool is **not used to introduce new topics beyond the syllabus** but rather to enhance the user's understanding of existing topics "
        "by providing deeper insights, definitions, and broader industry context. "
        "By combining official exam guidelines with validated external knowledge, the assistant ensures accurate, well-structured responses that aid in study material creation and practice question generation."
    ),
    tools=[query_tool, scrape_tool],
    llm=llm_agent,
    verbose=True,
)


In [55]:
exam_preparation_task = Task(
    description=(
        "Answer the user's AWS AI Practitioner exam query by first consulting the official exam syllabus and pattern using the query tool. "
        "Then, enhance the response by retrieving **additional context or explanations** about the topic from trusted certification-related sources using the scrape tool. "
        "The scrape tool does **not introduce topics beyond the syllabus** but only provides deeper insights into existing topics to reinforce understanding. "
        "Based on the gathered information, generate study materials, practice questions, or structured explanations tailored to the user's exam preparation needs."
    ),
    expected_output="A well-structured response that aligns with the official exam syllabus while providing additional validated insights for better understanding.",
    agent=aws_ai_exam_agent,
)


In [56]:
# Run Crew with User Prompt
user_prompt = input("Enter your exam preparation query: ")
result = aws_ai_exam_assistant.kickoff(inputs={"user_prompt": user_prompt})

print("**********************************************************************")
print(result)

Enter your exam preparation query: short note about this exam


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: ba4dffda-433d-4f57-b23a-8057ee4e5324                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🚀 Crew: crew
└── 📋 Task: 1a82ad01-cb42-4cd8-beb1-c5c5ba6cb0c7
       Status: Executing Task...

🚀 Crew: crew
└── 📋 Task: 1a82ad01-cb42-4cd8-beb1-c5c5ba6cb0c7
       Status: Executing Task...
    └── 🤖 Agent: AWS AI Practitioner Assistant
            Status: In Progress

# Agent: AWS AI Practitioner Assistant
## Task: Answer the user's query by first consulting the AWS AI Practitioner exam syllabus and exam pattern (via query_tool), and then scraping the web (via scrape_tool) to prapre the study material, practice questions, or any other information needed. as per the user need


...

🚀 Crew: crew
└── 📋 Task: 1a82ad01-cb42-4cd8-beb1-c5c5ba6cb0c7
       Status: Executing Task...
    └── 🤖 Agent: AWS AI Practitioner Assistant
            Status: ✅ Completed
        ├── 🔧 Used AWS AI practioner exam guide (9)
        └── 🔧 Used Read website content (19)

🚀 Crew: crew
└── 📋 Task: 1a82ad01-cb42-4cd8-beb1-c5c5ba6cb0c7
       Assigned to: AWS AI Practitioner Assistant
       Status: ✅ Completed
    └── 🤖 Agent: AWS AI Practitioner Assistant
            Status: ✅ Completed
        ├── 🔧 Used AWS AI practioner exam guide (9)
        └── 🔧 Used Read website content (19)

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 1a82ad01-cb42-4cd8-beb1-c5c5ba6cb0c7                                                                     │
│  Agent: AWS AI Practitioner Assistant                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: ba4dffda-433d-4f57-b23a-8057ee4e5324                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**********************************************************************
Based on the AWS AI Practitioner exam syllabus, the exam covers content domains such as Fundamentals of AI and ML, Fundamentals of Generative AI, Applications of Foundation Models, Guidelines for Responsible AI, and Security, Compliance, and Governance for AI Solutions. The exam pattern includes various question types like multiple choice, multiple response, ordering, matching, and case study questions. Additionally, you can access example websites for additional study materials and practice questions.


In [64]:
import textwrap

# Convert result to a string
result_str = str(result)  # Ensures that 'result' is a string

wrapped_text = textwrap.fill(result_str, width=80)  # Wrap text at 80 characters
print(wrapped_text)


Based on the AWS AI Practitioner exam syllabus, the exam covers content domains
such as Fundamentals of AI and ML, Fundamentals of Generative AI, Applications
of Foundation Models, Guidelines for Responsible AI, and Security, Compliance,
and Governance for AI Solutions. The exam pattern includes various question
types like multiple choice, multiple response, ordering, matching, and case
study questions. Additionally, you can access example websites for additional
study materials and practice questions.
